# Optunaを用いたパラメタの最適化

## データセットの生成

### 関数の定義

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def generate_uncorrelated_gmm_2d_toy_dataset(num_total_samples=2000,
                                            ratio_mode1=0.5, # モード1のデータが全体の何割を占めるか (0.0 ~ 1.0)
                                            mu1=np.array([-3.0, -3.0]),
                                            sigma1_diag=np.array([1.0, 1.0]), # 対角成分のみ指定
                                            mu2=np.array([3.0, 3.0]),
                                            sigma2_diag=np.array([1.0, 1.0]), # 対角成分のみ指定
                                            random_seed=42):
    """
    2つの無相関な2次元正規分布を組み合わせたガウス混合モデル (GMM) からトイデータセットを生成します。
    各正規分布の比率を指定できます。

    Args:
        num_total_samples (int): 生成する総サンプル数。
        ratio_mode1 (float): モード1 (1つ目の正規分布) のデータが全体の何割を占めるか (0.0 ~ 1.0)。
                              モード2の比率は (1 - ratio_mode1) になります。
        mu1 (np.ndarray): 1つ目の正規分布の平均ベクトル (2次元)。
        sigma1_diag (np.ndarray): 1つ目の正規分布の共分散行列の対角成分 (2要素)。
                                  非対角成分は0と仮定されます。
        mu2 (np.ndarray): 2つ目の正規分布の平均ベクトル (2次元)。
        sigma2_diag (np.ndarray): 2つ目の正規分布の共分散行列の対角成分 (2要素)。
                                  非対角成分は0と仮定されます。
        random_seed (int): 乱数生成のシード。

    Returns:
        np.ndarray: 生成されたデータポイント (N x 2)。
    """
    if not (0.0 <= ratio_mode1 <= 1.0):
        raise ValueError("ratio_mode1 must be between 0.0 and 1.0")

    np.random.seed(random_seed)

    # 各モードのサンプル数を計算
    num_samples_mode1 = int(num_total_samples * ratio_mode1)
    num_samples_mode2 = num_total_samples - num_samples_mode1 # 残りはモード2

    # 共分散行列を作成 (対角成分のみ有効)
    sigma1 = np.diag(sigma1_diag)
    sigma2 = np.diag(sigma2_diag)

    print(f"Generating {num_total_samples} samples:")
    print(f"  Mode 1 ({ratio_mode1*100:.1f}%): {num_samples_mode1} samples")
    print(f"  Mode 2 ({(1-ratio_mode1)*100:.1f}%): {num_samples_mode2} samples")

    # 1つ目の正規分布からデータを生成
    data_mode1 = np.random.multivariate_normal(mu1, sigma1, num_samples_mode1)

    # 2つ目の正規分布からデータを生成
    data_mode2 = np.random.multivariate_normal(mu2, sigma2, num_samples_mode2)

    # 両方のデータを結合
    dataset = np.vstack((data_mode1, data_mode2))

    # データをシャッフル (モードの偏りをなくすため)
    np.random.shuffle(dataset)

    return dataset

def plot_2d_dataset(dataset, title="2D GMM Toy Dataset (Uncorrelated)"):
    """
    2次元データセットをプロットします。

    Args:
        dataset (np.ndarray): プロットするデータセット (N x 2)。
        title (str): グラフのタイトル。
    """
    plt.figure(figsize=(8, 6))
    plt.scatter(dataset[:, 0], dataset[:, 1], s=10, alpha=0.7)
    plt.title(title)
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.grid(True)
    plt.axis('equal') # x軸とy軸のスケールを合わせる
    plt.show()

### 関数の実行

In [ ]:
# 例4: 比率30:70、円形クラスタ
print("--- Example 4: 30:70 Ratio, Circular Clusters ---")
dataset4 = generate_uncorrelated_gmm_2d_toy_dataset(
    num_total_samples=2000,
    ratio_mode1=0.3,
    mu1=np.array([-3.0, -3.0]),
    sigma1_diag=np.array([1.0, 1.0]),
    mu2=np.array([3.0, 3.0]),
    sigma2_diag=np.array([1.0, 1.0])
)
plot_2d_dataset(dataset4, title="Uncorrelated GMM (30:70, Circular)")

## NNと学習データ、ハイパーパラメータ、optunaなどの準備

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import optuna
import matplotlib.pyplot as plt
from torch.optim import Adam

# --- 時間埋め込み（正弦波位置エンコーディング）---
def pos_encoding(timesteps, output_dim, device='cpu'):
    position = timesteps.unsqueeze(1).float().to(device)
    div_term = torch.exp(torch.arange(0, output_dim, 2, device=device, dtype=torch.float32) * (-np.log(10000.0) / output_dim))
    sin_vals = torch.sin(position * div_term)
    cos_vals = torch.cos(position * div_term)
    sinusoid = torch.cat([sin_vals, cos_vals], dim=1)

    if sinusoid.shape[1] < output_dim:
        padding = torch.zeros(sinusoid.shape[0], output_dim - sinusoid.shape[1], device=device, dtype=torch.float32)
        sinusoid = torch.cat([sinusoid, padding], dim=1)
    elif sinusoid.shape[1] > output_dim:
        sinusoid = sinusoid[:, :output_dim]
    return sinusoid

# --- 拡散モデル ---
# 層数を可変にするため、fc2, fc3, bn2, bn3, dropout2, dropout3 をリスト化して扱う
class DiffusionModel(nn.Module):
    def __init__(self, input_data_dim, num_layers, time_embed_dim=16, hidden_dim=256, dropout_rate=0.1, activation='leaky_relu'):
        super(DiffusionModel, self).__init__()
        self.time_embed_dim = time_embed_dim
        self.input_data_dim = input_data_dim
        self.num_layers = num_layers # 隠れ層の数

        if activation == 'leaky_relu':
            self.activation_fn = nn.LeakyReLU()
        elif activation == 'elu':
            self.activation_fn = nn.ELU()
        elif activation == 'gelu':
            self.activation_fn = nn.GELU()
        else:
            self.activation_fn = nn.ReLU() # デフォルトはReLU

        layers = []
        # 最初の層
        layers.append(nn.Linear(self.input_data_dim + time_embed_dim, hidden_dim))
        layers.append(nn.BatchNorm1d(hidden_dim))
        layers.append(self.activation_fn)
        layers.append(nn.Dropout(dropout_rate))

        # 残りの隠れ層
        for _ in range(num_layers - 1): # num_layers は fc1 の後の隠れ層の数と解釈
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(self.activation_fn)
            layers.append(nn.Dropout(dropout_rate))

        self.hidden_layers = nn.Sequential(*layers)

        # 最終出力層
        self.output_layer = nn.Linear(hidden_dim, self.input_data_dim)

    def forward(self, x, t):
        t_embed = pos_encoding(t, self.time_embed_dim, x.device)
        x_t = torch.cat([x, t_embed], dim=-1)

        x_t = self.hidden_layers(x_t)
        
        return self.output_layer(x_t)

# --- 拡散プロセス ---
class Diffuser:
    def __init__(self, num_timesteps=1000, beta_start=0.0001, beta_end=0.02, device='cpu'):
        self.num_timesteps = num_timesteps
        self.device = device
        self.betas = torch.linspace(beta_start, beta_end, num_timesteps, device=device)
        self.alphas = 1 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)

    def add_noise(self, x_0, t):
        t_idx = t - 1
        alpha_bar = self.alpha_bars.gather(0, t_idx).view(-1, 1)
        noise = torch.randn_like(x_0, device=self.device)
        x_t = torch.sqrt(alpha_bar) * x_0 + torch.sqrt(1 - alpha_bar) * noise
        return x_t, noise

    def denoise(self, model, x, t):
        T = self.num_timesteps
        assert torch.all(t >= 1) and torch.all(t <= T)
        t_idx = t - 1
        alpha = self.alphas.gather(0, t_idx).view(-1, 1)
        alpha_bar = self.alpha_bars.gather(0, t_idx).view(-1, 1)

        model.eval()
        with torch.no_grad():
            eps = model(x, t)

        mu = (x - (1 - alpha) / torch.sqrt(1 - alpha_bar) * eps) / torch.sqrt(alpha)
        sigma = torch.sqrt(self.betas.gather(0, t_idx)).view(-1, 1)
        z = torch.randn_like(x, device=self.device)
        z[t == 1] = 0

        x_prev = mu + sigma * z
        return x_prev

# --- データセットの準備 ---
# 仮のデータセットを生成
def get_dummy_dataset(num_samples=1000, data_dim=1, seed=42):
    np.random.seed(seed)
    # ガウス分布データ
    data = np.random.randn(num_samples, data_dim).astype(np.float32)
    # 少しだけ特徴を付ける (例: 2つのクラスター)
    data[:num_samples//2] += 2
    data[num_samples//2:] -= 2
    return torch.from_numpy(data)

# データセットのロード（実際のデータセットに置き換えてください）
dataset4 = generate_uncorrelated_gmm_2d_toy_dataset(
    num_total_samples=2000,
    ratio_mode1=0.3,
    mu1=np.array([-3.0, -3.0]),
    sigma1_diag=np.array([1.0, 1.0]),
    mu2=np.array([3.0, 3.0]),
    sigma2_diag=np.array([1.0, 1.0])
)
# dataset4 = get_dummy_dataset(seed=42) # 仮のデータセット
input_data_dim = dataset4.shape[1] # 

# その他のハイパーパラメータ（Optunaで探索しないもの）
num_timesteps = 1000
epochs_per_trial = 20 # Optunaの1試行あたりのエポック数
device = 'cuda' if torch.cuda.is_available() else 'cpu'
time_embed_dim = 16
batch_size = 64
diffuser = Diffuser(num_timesteps=num_timesteps, device=device)


# --- Optunaの目的関数を定義 ---
def objective(trial):
    # 探索するハイパーパラメータを定義
    # ノード数 (hidden_dim): 32から512までを2の冪乗で探索
    hidden_dim = trial.suggest_categorical('hidden_dim', [32, 64, 128, 256, 512])

    # 層数 (num_layers): 1から4までの整数で探索 (fc1の後の隠れ層の数)
    # total_layers = num_layers + 1 (入力層直後 + 隠れ層数 + 出力層直前)
    num_layers = trial.suggest_int('num_layers', 1, 4) # 隠れ層の数 (fc1以外)

    # 学習率 (lr): 1e-5から1e-2までを対数スケールで探索
    lr = trial.suggest_loguniform('lr', 1e-5, 1e-2)

    # ドロップアウト率 (dropout_rate): 0.0から0.5までをステップ0.1で探索
    dropout_rate = trial.suggest_float('dropout_rate', 0.0, 0.5, step=0.1)

    # 活性化関数 (activation):
    activation = trial.suggest_categorical('activation', ['relu', 'leaky_relu', 'elu', 'gelu'])


    # モデルの構築
    # input_data_dim はグローバル変数または引数で渡す
    model = DiffusionModel(
        input_data_dim=input_data_dim,
        num_layers=num_layers,
        time_embed_dim=time_embed_dim,
        hidden_dim=hidden_dim,
        dropout_rate=dropout_rate,
        activation=activation
    ).to(device)

    optimizer = Adam(model.parameters(), lr=lr)

    # データローダー
    dataloader = torch.utils.data.DataLoader(dataset4, batch_size=batch_size, shuffle=True)

    # 学習ループ
    # 検証セットがあれば、ここで検証損失を計算し、それを返す
    # ここでは訓練損失を評価指標とします（簡易化のため）
    for epoch in range(epochs_per_trial):
        model.train() # モデルを訓練モードに設定
        loss_sum = 0.0
        for batch in dataloader:
            optimizer.zero_grad()
            x = batch.to(device).float()
            t = torch.randint(1, num_timesteps + 1, (len(x),), device=device)

            x_noisy, noise = diffuser.add_noise(x, t)
            x_noisy = x_noisy.float()
            noise = noise.float()
            noise_pred = model(x_noisy, t)
            loss = F.mse_loss(noise_pred, noise)

            loss.backward()
            optimizer.step()
            loss_sum += loss.item()

        avg_loss = loss_sum / len(dataloader)

        # Optunaの早期停止 (Pruning)
        # 訓練中に有望でない試行を早期に打ち切ることで、全体の探索時間を短縮
        # 例えば、現在の試行がこれまでのベストな試行よりも著しく悪い場合など
        trial.report(avg_loss, epoch) # 損失と現在のエポック数を報告
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    # 最終的な評価指標を返す（ここでは最後のEpochの訓練損失）
    return avg_loss

# --- Optunaの最適化実行 ---
if __name__ == "__main__":
    # Studyの作成
    # direction='minimize' は目的関数が返す値を最小化することを目指す
    # sampler は探索アルゴリズム (TPESamplerがデフォルトで推奨)
    # pruner は早期停止のメカニズム (MedianPrunerが一般的)
    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=42), # 再現性のためにseedを設定
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5, interval_steps=1)
    )

    # 最適化の実行
    # n_trials は試行回数
    # timeout は最適化の最大時間（秒）
    print("Optuna search started...")
    study.optimize(objective, n_trials=50, timeout=600) # 50回試行、最大10分

    print("\nOptuna search finished.")
    print("Best trial:")
    print(f"  Value: {study.best_value}") # 最も良かった試行の評価値
    print(f"  Params: {study.best_params}") # 最も良かった試行のハイパーパラメータ

In [ ]:
# 結果の可視化 (Optunaの可視化機能)
# 注意: これらはウェブブラウザでインタラクティブなグラフを生成します。
# jupyter notebook などで実行すると表示されます。
# plt.plot() ではなく、Optunaのplot_ functions を使います。
import plotly.io as pio # plotlyが必要な場合
pio.renderers.default = "notebook" # jupyterの場合の設定

# ハイパーパラメータの重要度
fig = optuna.visualization.plot_param_importances(study)
fig.show()

# 最適化履歴
fig = optuna.visualization.plot_optimization_history(study)
fig.show()

# ハイパーパラメータ間の関係 (個々のハイパーパラメータがどの値を取ったか)
fig = optuna.visualization.plot_slice(study)
fig.show()

# 最適化されたハイパーパラメータの分布
fig = optuna.visualization.plot_parallel_coordinate(study)
fig.show()